# 01 - Data cleaning & exploration

Explore the raw sales history, check data quality and build model features.
Works with the bundled synthetic data (`data/raw/sample_sales.csv`) or the Kaggle
*Store Item Demand Forecasting Challenge* `train.csv` (`date, store, item, sales`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
import pandas as pd

from src.forecasting.data import FEATURES, build_features, load_sales

raw = pd.read_csv(ROOT / "data/raw/sample_sales.csv")
raw.head()

## Data quality checks

In [ ]:
print("rows:", len(raw))
print("missing values:\n", raw.isna().sum())
print("negative sales:", (raw["sales"] < 0).sum())
print("duplicate (sku, date):", raw.duplicated(["sku", "date"]).sum())

`load_sales` parses dates, aggregates duplicates, fills missing days with 0 and clips negatives.

In [ ]:
sales = load_sales(ROOT / "data/raw/sample_sales.csv")
sales.groupby("sku")["sales"].describe()

## Seasonality

In [ ]:
weekday = sales.assign(dow=sales["date"].dt.day_name()).groupby("dow")["sales"].mean()
monthly = sales.assign(month=sales["date"].dt.month).groupby("month")["sales"].mean()
print(weekday.sort_values(ascending=False).round(1))
print(monthly.round(1))

## Feature engineering
Lags (1, 7, 14, 28 days), rolling means/std of *past* values only (no leakage) and calendar features.

In [ ]:
features = build_features(sales, sorted(sales["sku"].unique()))
features[["date", "sku", "sales", *FEATURES]].head()

In [ ]:
out = ROOT / "data/processed/features.csv"
features.to_csv(out, index=False)
print(f"saved {len(features):,} rows -> {out}")